# Notebook 00: Setup & Connection Testing

## Overview
This notebook sets up the environment and validates all connections needed for the RAG demo:

1. **Install Dependencies**: Required Python packages
2. **Download Dataset**: HuggingFace E-commerce FAQ dataset
3. **Test AWS Connection**: Verify boto3 credentials
4. **Test Bedrock API**: Test Claude Sonnet 4 and Titan Embeddings
5. **Environment Setup**: Configure credentials and settings

---

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q boto3 datasets pandas python-dotenv matplotlib seaborn

In [ ]:
# Import libraries
import boto3
import json
import os
import pandas as pd
from datasets import load_dataset
from botocore.exceptions import ClientError, NoCredentialsError
from dotenv import load_dotenv

print("[OK] All libraries imported successfully!")

---

## 2. Configure AWS Credentials

**Options to set credentials:**

### Option 1: Create `.env` file (Recommended)
Create a `.env` file in the project root:
```bash
AWS_ACCESS_KEY_ID=your_access_key_here
AWS_SECRET_ACCESS_KEY=your_secret_key_here
AWS_DEFAULT_REGION=us-east-1
```

### Option 2: Set environment variables in notebook
```python
os.environ['AWS_ACCESS_KEY_ID'] = 'your_access_key'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your_secret_key'
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
```

### Option 3: Use AWS CLI configure
Run in terminal: `aws configure`

In [ ]:
# Load environment variables from .env file (if exists)
load_dotenv()

# Check if credentials are set
aws_access_key = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_key = os.getenv('AWS_SECRET_ACCESS_KEY')
aws_region = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')

if aws_access_key and aws_secret_key:
    print(f"[OK] AWS credentials loaded")
    print(f"   Region: {aws_region}")
    print(f"   Access Key: {aws_access_key[:8]}...")
else:
    print("[WARNING] AWS credentials NOT found!")
    print("Please set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY")
    print("\nYou can set them here:")
    # Uncomment and fill in your credentials:
    # os.environ['AWS_ACCESS_KEY_ID'] = 'AKIA...'
    # os.environ['AWS_SECRET_ACCESS_KEY'] = 'wJalr...'
    # os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

---

## 3. Test AWS Connection

Verify that boto3 can connect to AWS using your credentials.

In [ ]:
def test_aws_connection():
    """Test basic AWS connection using STS."""
    try:
        # Create STS client
        sts = boto3.client('sts', region_name=aws_region)
        
        # Get caller identity
        response = sts.get_caller_identity()
        
        print("[OK] AWS Connection Successful!")
        print(f"   Account ID: {response['Account']}")
        print(f"   User ARN: {response['Arn']}")
        print(f"   User ID: {response['UserId']}")
        return True
        
    except NoCredentialsError:
        print("[FAILED] AWS credentials not found!")
        print("   Please configure your AWS credentials.")
        return False
        
    except ClientError as e:
        print(f"[FAILED] AWS connection failed: {e}")
        return False
    
    except Exception as e:
        print(f"[FAILED] Unexpected error: {e}")
        return False

# Test connection
aws_connected = test_aws_connection()

---

## 4. Test Bedrock Model Access

### Step 1: Check Available Models

In [ ]:
def list_available_bedrock_models():
    """List all available Bedrock foundation models."""
    try:
        bedrock = boto3.client('bedrock', region_name=aws_region)
        
        response = bedrock.list_foundation_models()
        
        print("[OK] Available Bedrock Models:\n")
        
        # Filter for Claude and Titan models
        claude_models = []
        titan_models = []
        
        for model in response['modelSummaries']:
            model_id = model['modelId']
            model_name = model['modelName']
            
            if 'claude' in model_id.lower():
                claude_models.append((model_id, model_name))
            elif 'titan' in model_id.lower() and 'embed' in model_id.lower():
                titan_models.append((model_id, model_name))
        
        print(" Claude Models (LLM):")
        for model_id, name in claude_models:
            print(f"   - {model_id}")
        
        print("\n Titan Embedding Models:")
        for model_id, name in titan_models:
            print(f"   - {model_id}")
        
        return True
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'AccessDeniedException':
            print("[FAILED] Access Denied to Bedrock")
            print("   Please ensure your IAM user has Bedrock permissions.")
        else:
            print(f"[FAILED] Error listing models: {e}")
        return False
    
    except Exception as e:
        print(f"[FAILED] Unexpected error: {e}")
        return False

if aws_connected:
    list_available_bedrock_models()
else:
    print("[WARNING] Skipping - AWS not connected")

### Step 2: Test Claude Sonnet 4 (LLM)

In [ ]:
def test_claude_sonnet():
    """Test Claude Sonnet 4 API."""
    try:
        bedrock_runtime = boto3.client('bedrock-runtime', region_name=aws_region)
        
        # Model ID for Claude 3.5 Sonnet v2
        model_id = 'anthropic.claude-3-5-sonnet-20241022-v2:0'
        
        # Test prompt
        prompt = "Say 'Hello! I am Claude Sonnet 4 running via AWS Bedrock.' in one sentence."
        
        # Prepare request
        request_body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 100,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0.1
        }
        
        print(f" Testing Claude Sonnet 4...")
        print(f"   Model ID: {model_id}")
        print(f"   Prompt: {prompt}")
        print()
        
        # Invoke model
        response = bedrock_runtime.invoke_model(
            modelId=model_id,
            body=json.dumps(request_body)
        )
        
        # Parse response
        response_body = json.loads(response['body'].read())
        response_text = response_body['content'][0]['text']
        
        print("[OK] Claude Sonnet 4 Response:")
        print(f"   {response_text}")
        print()
        print(f"   Input tokens: {response_body.get('usage', {}).get('input_tokens', 'N/A')}")
        print(f"   Output tokens: {response_body.get('usage', {}).get('output_tokens', 'N/A')}")
        
        return True
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'AccessDeniedException':
            print("[FAILED] Access Denied to Claude Sonnet 4")
            print("   Please request model access in Bedrock console:")
            print("   https://console.aws.amazon.com/bedrock/home#/modelaccess")
        else:
            print(f"[FAILED] Error: {e}")
        return False
    
    except Exception as e:
        print(f"[FAILED] Unexpected error: {e}")
        return False

if aws_connected:
    claude_works = test_claude_sonnet()
else:
    print("[WARNING] Skipping - AWS not connected")
    claude_works = False

### Step 3: Test Titan Embeddings

In [ ]:
def test_titan_embeddings():
    """Test Amazon Titan Embeddings API."""
    try:
        bedrock_runtime = boto3.client('bedrock-runtime', region_name=aws_region)
        
        # Model ID for Titan Embeddings V2
        model_id = 'amazon.titan-embed-text-v2:0'
        
        # Test text
        test_text = "What is your return policy?"
        
        # Prepare request
        request_body = {
            "inputText": test_text
        }
        
        print(f" Testing Titan Embeddings...")
        print(f"   Model ID: {model_id}")
        print(f"   Input: {test_text}")
        print()
        
        # Invoke model
        response = bedrock_runtime.invoke_model(
            modelId=model_id,
            body=json.dumps(request_body)
        )
        
        # Parse response
        response_body = json.loads(response['body'].read())
        embedding = response_body['embedding']
        
        print("[OK] Titan Embeddings Response:")
        print(f"   Embedding dimensions: {len(embedding)}")
        print(f"   Sample values (first 5): {embedding[:5]}")
        print(f"   Embedding range: [{min(embedding):.4f}, {max(embedding):.4f}]")
        
        return True
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'AccessDeniedException':
            print("[FAILED] Access Denied to Titan Embeddings")
            print("   Please request model access in Bedrock console:")
            print("   https://console.aws.amazon.com/bedrock/home#/modelaccess")
        else:
            print(f"[FAILED] Error: {e}")
        return False
    
    except Exception as e:
        print(f"[FAILED] Unexpected error: {e}")
        return False

if aws_connected:
    titan_works = test_titan_embeddings()
else:
    print("[WARNING] Skipping - AWS not connected")
    titan_works = False

---

## 5. Download E-commerce FAQ Dataset

Download the HuggingFace dataset and save it locally.

In [ ]:
def download_ecommerce_faq():
    """Download and explore HuggingFace Ecommerce FAQ dataset."""
    try:
        print(" Downloading E-commerce FAQ dataset...")
        
        # Load dataset
        dataset = load_dataset("Andyrasika/Ecommerce_FAQ")
        
        print("\n[OK] Dataset loaded successfully!")
        print(f"\n{dataset}")
        
        # Convert to pandas for easier viewing
        df = dataset['train'].to_pandas()
        
        print(f"\n Dataset Statistics:")
        print(f"   Total Q&A pairs: {len(df)}")
        print(f"   Columns: {list(df.columns)}")
        print(f"   Average question length: {df['question'].str.len().mean():.1f} characters")
        print(f"   Average answer length: {df['answer'].str.len().mean():.1f} characters")
        
        # Save to CSV
        output_path = '../data/ecommerce_faq.csv'
        df.to_csv(output_path, index=False)
        print(f"\n Dataset saved to: {output_path}")
        
        return df
        
    except Exception as e:
        print(f"[FAILED] Error downloading dataset: {e}")
        return None

# Download dataset
faq_df = download_ecommerce_faq()

### Preview Dataset

In [ ]:
if faq_df is not None:
    print(" Sample Q&A Pairs:\n")
    
    # Show first 5 rows
    for idx, row in faq_df.head(5).iterrows():
        print(f"{'='*80}")
        print(f"Q{idx+1}: {row['question']}")
        print(f"A{idx+1}: {row['answer']}")
        print()

In [ ]:
# Display full dataframe
if faq_df is not None:
    display(faq_df.head(10))

---

## 6. Connection Summary

Summary of all connection tests.

In [ ]:
print("="*80)
print(" SETUP & CONNECTION TEST SUMMARY")
print("="*80)
print()

# Check status
status = {
    "AWS Connection": "[OK] Connected" if aws_connected else "[FAILED] Failed",
    "Claude Sonnet 4": "[OK] Working" if claude_works else "[FAILED] Not Available",
    "Titan Embeddings": "[OK] Working" if titan_works else "[FAILED] Not Available",
    "FAQ Dataset": "[OK] Downloaded" if faq_df is not None else "[FAILED] Failed"
}

for service, state in status.items():
    print(f"{service:.<30} {state}")

print()
print("="*80)

# Overall status
all_working = all([aws_connected, claude_works, titan_works, faq_df is not None])

if all_working:
    print("\n SUCCESS! All connections working. Ready to proceed with RAG demo!")
    print("\n Next Step: Run Notebook 01 - Data Exploration & Chunking Strategies")
else:
    print("\n[WARNING] Some connections failed. Please fix the issues above before proceeding.")
    print("\n Common Issues:")
    if not aws_connected:
        print("   - AWS: Check your access keys and region")
    if not claude_works or not titan_works:
        print("   - Bedrock: Request model access at https://console.aws.amazon.com/bedrock/home#/modelaccess")
    if faq_df is None:
        print("   - Dataset: Check internet connection")

---

## 7. Save Configuration (Optional)

Save validated settings for use in other notebooks.

In [ ]:
if all_working:
    config = {
        "aws_region": aws_region,
        "claude_model_id": "anthropic.claude-3-5-sonnet-20241022-v2:0",
        "titan_embed_model_id": "amazon.titan-embed-text-v2:0",
        "faq_dataset_path": "../data/ecommerce_faq.csv",
        "embedding_dimensions": 1024  # Titan v2 dimensions
    }
    
    config_path = '../data/config.json'
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    
    print(f"[OK] Configuration saved to: {config_path}")
    print("\nConfig contents:")
    print(json.dumps(config, indent=2))

---

## [OK] Notebook Complete

You've successfully:
- [OK] Installed all required packages
- [OK] Configured AWS credentials
- [OK] Tested AWS connection
- [OK] Verified Bedrock Claude Sonnet 4 access
- [OK] Verified Bedrock Titan Embeddings access
- [OK] Downloaded E-commerce FAQ dataset (79 Q&A pairs)
- [OK] Saved configuration for next notebooks

**Next**: Proceed to `01_data_exploration_chunking.ipynb` to explore chunking strategies!